# NIG-TimeNet: the evaluation battery - can this model be trusted?

**Runtime > Run all.** This notebook judges a FINISHED run - it trains nothing. It auto-detects the
latest run your `ar_stream` / `panel_ar_stream` notebooks saved to Drive (the prediction tables they
write at the end) and interrogates it from ten angles, the way a trader would before trusting money to it:

1. **Sanity** - does the predicted magnitude rank the true moves (IC), and is the direction unbiased?
2. **Confidence calibration** - is the model right more often exactly where it is confident?
3. **Anti-cowardice** - does it predict real magnitudes, or hide near zero?
4. **Trading grid** - trade the predicted direction, top-% by magnitude, with cost.
5. **Realistic execution** - limit orders AT the predicted price, market fallback.
6. **Uncertainty filter** - do low-sigma (self-confident) predictions do better?
7. **Shock regimes** - calm to wildest quintile of the true move: where the edge lives.
8. **Weekly stability** - is the edge there week after week, or one lucky stretch?
9. **Per-coin** - which coins carry the edge and which drown it.
10. **Flip test** - is the direction inverted?

Then a verdict that weighs the best strategy's Sharpe, the shock sanity and the weekly consistency together.


In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "feat/ar-stream"
EXPECT_DEVICE = "cuda"                         # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = ()                             # torch/pandas/numpy ship with Colab
COST = 0.0008                                  # round-trip cost per trade, as a fraction of notional


## 1. Setup
The first cell is plain code: nothing of the package exists on the VM yet, so it downloads the one stdlib-only file
`colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.


In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))


In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)


In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)


## 2. The run: auto-detect the latest on Drive
The newest folder under `DRIVE_RUNS` that carries `predictions/test_predictions.parquet` is the run under
judgment: its VAL and TEST tables load as-is (one row per window/coin, step).


In [ ]:
from pathlib import Path
import pandas as pd
found = sorted(Path(DRIVE_RUNS).glob("*/predictions/test_predictions.parquet"),
               key=lambda p: p.stat().st_mtime)
assert found, f"no run with predictions under {DRIVE_RUNS}: finish an ar_stream/panel_ar_stream run first"
RUN = found[-1].parent.parent
V = pd.read_parquet(RUN / "predictions/val_predictions.parquet")
T = pd.read_parquet(RUN / "predictions/test_predictions.parquet")
print("run:", RUN.name)
print(f"VAL n={len(V):,}   TEST n={len(T):,}",
      "coins:", sorted(T["coin"].unique()) if "coin" in T else "single-coin run")


## 3. The battery: ten angles, then the verdict
Every section prints its table plus a one-line reading. The final verdict names the best realistic
strategy (TEST Sharpe) but tells you to check the shock and weekly sections before believing it.


In [ ]:
from nig_timenet_torch.eval.battery import full_battery
out = full_battery(V, T, cost=COST)
for name, section in out.items():
    print("=" * 100)
    print(section["title"])
    for row in section["rows"]:
        print(row)
    if section.get("note"):
        print("->", section["note"])


## 4. How to read it
* An IC near zero in section 1 means the magnitude carries no ranking information - only the direction can pay.
* Section 3's ratio well under 1 is the coward signature the czar loss is built against; well over 1 means the
  model overshoots and the limit orders in section 5 will rarely fill.
* Section 7 answers the owner's core question: the accuracy is EXPECTED to decay in the wildest quintile
  (news, thin liquidity, squeezes are not pattern); what matters is that the PnL does not explode there.
* The verdict's thresholds (Sharpe 0.10 / 0.05 per trade) are per-trade, not annualized: read them as a
  ranking across strategies, and demand the weekly section agree before calling anything an edge.
